## Initialization Steps

- set up the transformers library
- get some nice tools for doing things with strings


In [ ]:
from transformers import pipeline
import string

# Initialize once
unmasker = pipeline(
    "fill-mask",
    model="google-bert/bert-base-uncased"
)

# Simple stopword set (extend freely)
STOPWORDS = {
    "the", "a", "an", "of", "and", "to", "in", "on",
    "for", "with", "at", "by", "from", "as", "is"
}

def complete_mask(sentence, n=5, extra_stopwords=None):
    """
    Return top-n filtered BERT mask completions.
    Removes punctuation, WordPiece fragments, and stopwords.
    """

    if "[MASK]" not in sentence:
        raise ValueError("Input sentence must contain '[MASK]' token.")

    if extra_stopwords is None:
        extra_stopwords = set()

    stopwords = STOPWORDS.union(extra_stopwords)

    raw_results = unmasker(sentence, top_k=4*n)

    filtered = []

    for r in raw_results:
        token = r["token_str"].strip().lower()

        # remove subword fragments
        if token.startswith("##"):
            continue

        # remove punctuation-only tokens
        if all(c in string.punctuation for c in token):
            continue

        # remove stopwords
        if token in stopwords:
            continue

        # keep alphabetic tokens only
        if not token.isalpha():
            continue

        filtered.append({
            "completion": token,
            "score": r["score"]
        })

        if len(filtered) == n:
            break

    return filtered

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
examples = [
    "Hamilton College is located in [MASK], New York.",
    "Linear algebra is one of the most [MASK] subjects in mathematics.",
    "The theorem follows immediately from the [MASK].",
    "The man went to work. He is a [MASK].",
    "The woman went to work. She is a [MASK]."
]

In [46]:
for sentence in examples:
    output = complete_mask(sentence, 10)
    for o in output:
        print(f"{sentence} {o['completion']:>12} score={o['score']:.4f}")

Hamilton College is located in [MASK], New York.     hamilton score=0.6242
Hamilton College is located in [MASK], New York.      buffalo score=0.0864
Hamilton College is located in [MASK], New York.    rochester score=0.0720
Hamilton College is located in [MASK], New York.     brooklyn score=0.0354
Hamilton College is located in [MASK], New York.       albany score=0.0217
Hamilton College is located in [MASK], New York.     syracuse score=0.0216
Hamilton College is located in [MASK], New York.       queens score=0.0109
Hamilton College is located in [MASK], New York.       ithaca score=0.0080
Hamilton College is located in [MASK], New York.      niagara score=0.0079
Hamilton College is located in [MASK], New York.       geneva score=0.0060
Linear algebra is one of the most [MASK] subjects in mathematics.      popular score=0.3702
Linear algebra is one of the most [MASK] subjects in mathematics.      studied score=0.2492
Linear algebra is one of the most [MASK] subjects in mathematics. 

In [ ]:
complete_mask("The [MASK] is the best animal.",5) #try adding "A" to the front

: 

In [66]:
complete_mask("The Dalai Lama likes [MASK] philosophy.",10)

[{'completion': 'this', 'score': 0.10563056915998459},
 {'completion': 'his', 'score': 0.08561185747385025},
 {'completion': 'buddhist', 'score': 0.07766035199165344},
 {'completion': 'western', 'score': 0.05020971968770027},
 {'completion': 'chinese', 'score': 0.04134850576519966},
 {'completion': 'indian', 'score': 0.024882035329937935},
 {'completion': 'political', 'score': 0.01750977896153927},
 {'completion': 'hindu', 'score': 0.016128310933709145},
 {'completion': 'modern', 'score': 0.01576649770140648},
 {'completion': 'that', 'score': 0.012761092744767666}]